## Cell 1: Imports & Configuration

In [11]:
from __future__ import annotations

import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from openai import OpenAI

# -------- User-editable inputs --------
USER_QUERY = "Internal web app for the finance team with a relational database backend"

# Where to cache the raw ARG dump
ARG_CACHE_PATH = Path("../../plugin/skills/azure-enterprise-infra-planner/scripts/arg_raw_output_all.json")

# Azure OpenAI settings
AOAI_BASE_URL    = os.environ.get("AZURE_OPENAI_BASE_URL", "https://workloads-assistant-aoai.openai.azure.com/openai/v1")
AOAI_DEPLOYMENT  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-mini")
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "preview")

# FP-Growth mining parameters
MIN_ABSOLUTE_COUNT = 4    # itemset must appear in >= this many RGs
MIN_SUPPORT_FLOOR = 0.02  # never go below this fractional support
MIN_ITEMSET_LEN = 2       # ignore single-item itemsets
MAX_ITEMSETS = 200         # cap emitted MFIs
MIN_TRANSACTIONS = 30     # below this, mining is unreliable

# Property aggregation settings
MAX_PROPERTY_DEPTH = 5

## Cell 2: Azure OpenAI Client

In [12]:
_credential = DefaultAzureCredential()
_token_provider = get_bearer_token_provider(_credential, "https://cognitiveservices.azure.com/.default")

aoai = OpenAI(
    base_url=AOAI_BASE_URL,
    api_key="placeholder",
    default_query={"api-version": AOAI_API_VERSION},
    default_headers={"Authorization": f"Bearer {_token_provider()}"},
)
print(f"Azure OpenAI client ready (deployment={AOAI_DEPLOYMENT}).")

Azure OpenAI client ready (deployment=gpt-5-mini).


## Cell 3: Cached ARG Fetch

In [13]:
ARG_URL = "https://management.azure.com/providers/Microsoft.ResourceGraph/resources?api-version=2022-10-01"

ARG_KQL = """
Resources
| where type !startswith "microsoft.portal/"
| where type !startswith "providers.test/"
| where isempty(managedBy)
| where not (tags contains "hidden-") and not (tags contains "link:")
| where resourceGroup !startswith "mc_"
    and resourceGroup !startswith "databricks-rg-"
    and resourceGroup !startswith "azurebackuprg_"
    and resourceGroup !startswith "defaultresourcegroup-"
    and resourceGroup != "networkwatcherrg"
| project id, name, type, kind, location, resourceGroup, subscriptionId, sku, identity, properties
""".strip()


def fetch_arg_resources() -> list[dict]:
    """Fetch all resources via Azure Resource Graph, with caching."""
    # Try loading from cache first
    if ARG_CACHE_PATH.exists():
        print(f"Loading cached ARG data from {ARG_CACHE_PATH} ...")
        raw = json.loads(ARG_CACHE_PATH.read_text(encoding="utf-8"))
        # Handle both list and dict-with-data formats
        if isinstance(raw, list):
            print(f"  Loaded {len(raw)} resources from cache (list format).")
            return raw
        if isinstance(raw, dict) and "data" in raw:
            data = raw["data"]
            if isinstance(data, list):
                print(f"  Loaded {len(data)} resources from cache (dict format).")
                return data
        # Fallback: treat dict values as lists of resources
        if isinstance(raw, dict):
            all_items = []
            for v in raw.values():
                if isinstance(v, list):
                    all_items.extend(v)
            if all_items:
                print(f"  Loaded {len(all_items)} resources from cache (merged dict values).")
                return all_items
        print("  Cache format not recognized, fetching from ARG ...")

    # Fetch from ARG with pagination
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    all_rows: list[dict] = []
    skip_token: str | None = None
    page = 0

    while True:
        page += 1
        body: dict[str, Any] = {"query": ARG_KQL, "options": {"$top": 1000}}
        if skip_token:
            body["options"]["$skipToken"] = skip_token

        resp = requests.post(ARG_URL, headers=headers, json=body, timeout=60)
        resp.raise_for_status()
        result = resp.json()

        rows = result.get("data", [])
        all_rows.extend(rows)
        print(f"  Page {page}: received {len(rows)} rows (total so far: {len(all_rows)})")

        skip_token = result.get("$skipToken")
        if not skip_token or not rows:
            break

    # Cache results
    ARG_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    ARG_CACHE_PATH.write_text(json.dumps(all_rows, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Cached {len(all_rows)} resources to {ARG_CACHE_PATH}")
    return all_rows


resources = fetch_arg_resources()
print(f"\nTotal resources: {len(resources)}")

  Page 1: received 1000 rows (total so far: 1000)
  Page 2: received 1000 rows (total so far: 2000)
  Page 3: received 1000 rows (total so far: 3000)
  Page 4: received 1000 rows (total so far: 4000)
  Page 5: received 1000 rows (total so far: 5000)
  Page 6: received 1000 rows (total so far: 6000)
  Page 7: received 1000 rows (total so far: 7000)
  Page 8: received 1000 rows (total so far: 8000)
  Page 9: received 1000 rows (total so far: 9000)
  Page 10: received 1000 rows (total so far: 10000)
  Page 11: received 1000 rows (total so far: 11000)
  Page 12: received 1000 rows (total so far: 12000)
  Page 13: received 1000 rows (total so far: 13000)
  Page 14: received 1000 rows (total so far: 14000)
  Page 15: received 1000 rows (total so far: 15000)
  Page 16: received 904 rows (total so far: 15904)
Cached 15904 resources to plugin\skills\azure-enterprise-infra-planner\scripts\arg_raw_output_all.json

Total resources: 15904


## Cell 4: Filter Noise

In [14]:
# ARM child types with >=2 slashes (e.g. "microsoft.compute/virtualmachines/extensions")
DROP_ARM_CHILD_TYPES = True

# Auto-created types (spawned by Azure as side effects)
AUTO_CREATED_TYPES = frozenset({
    "microsoft.alertsmanagement/smartdetectoralertrules",
    "microsoft.insights/actiongroups",
    "microsoft.alertsmanagement/prometheusrulegroups",
    "microsoft.security/automations",
    "microsoft.security/pricings",
    "microsoft.operationsmanagement/solutions",
    "microsoft.security/iotsecuritysolutions",
    "microsoft.network/networkwatchers",
    "microsoft.advisor/recommendations",
})

# Internal Microsoft 1P resource providers (prefix-matched)
INTERNAL_MS_RP_PREFIXES = (
    "microsoft.portalservices/",
    "microsoft.cloudtest/",
    "microsoft.hydra/",
    "microsoft.swiftlet/",
    "microsoft.compute/swiftlets",
    "microsoft.fairfieldgardens/",
    "microsoft.footprintmonitoring/",
    "microsoft.saashub/",
    "microsoft.visualstudio/",
)

# Auto-managed sub-resource types
AUTO_MANAGED_SUBRESOURCE_TYPES = frozenset({
    "microsoft.containerregistry/registries/replications",
    "microsoft.containerregistry/registries/webhooks",
    "microsoft.compute/capacityreservationgroups/capacityreservations",
    "microsoft.compute/hostgroups/hosts",
    "microsoft.compute/galleries/images/versions",
    "microsoft.network/networkmanagers/ipampools",
    "microsoft.network/networkmanagers/verifierworkspaces",
})

# Marketplace types
MARKETPLACE_TYPES = frozenset({
    "microsoft.solutions/applications",
    "microsoft.solutions/appliances",
    "microsoft.saas/resources",
    "microsoft.saashub/cloudservices",
})


def _is_noise(rtype: str) -> bool:
    """Check if a resource type is architectural noise."""
    if not rtype:
        return False
    if rtype in AUTO_CREATED_TYPES or rtype in AUTO_MANAGED_SUBRESOURCE_TYPES or rtype in MARKETPLACE_TYPES:
        return True
    if any(rtype.startswith(p) for p in INTERNAL_MS_RP_PREFIXES):
        return True
    return False


def _is_auto_created(rtype: str) -> bool:
    """Superset of _is_noise: also drops ARM child types for itemset mining."""
    if not rtype:
        return False
    if DROP_ARM_CHILD_TYPES and rtype.count("/") >= 2:
        return True
    return _is_noise(rtype)


# Summary of filtering
all_types = {(r.get("type") or "").lower() for r in resources if r.get("type")}
noise_types = {t for t in all_types if _is_noise(t)}
auto_created_types = {t for t in all_types if _is_auto_created(t)}

print(f"Total distinct resource types: {len(all_types)}")
print(f"Noise types (property aggregation filter): {len(noise_types)}")
print(f"Auto-created types (FIM transaction filter): {len(auto_created_types)}")
if noise_types:
    print(f"\nExample noise types: {sorted(noise_types)[:5]}")
child_types = {t for t in all_types if t.count('/') >= 2}
print(f"ARM child types (>=2 slashes): {len(child_types)}")

Total distinct resource types: 182
Noise types (property aggregation filter): 32
Auto-created types (FIM transaction filter): 51

Example noise types: ['microsoft.alertsmanagement/prometheusrulegroups', 'microsoft.alertsmanagement/smartdetectoralertrules', 'microsoft.cloudtest/accounts', 'microsoft.cloudtest/hostedpools', 'microsoft.cloudtest/images']
ARM child types (>=2 slashes): 29


## Cell 5: Property Aggregation

In [15]:
# Property leaf whitelist — only aggregate values for these leaf names (case-insensitive)
PROPERTY_LEAF_WHITELIST = frozenset({
    "location", "kind",
    "sku", "name", "tier", "family", "capacity", "size",
    "publicnetworkaccess", "restrictoutboundnetworkaccess",
    "publicnetworkaccessforingestion", "publicnetworkaccessforquery",
    "defaultaction", "bypass",
    "disablelocalauth", "enablerbacauthorization",
    "minimumtlsversion", "minimaltlsversion",
    "identity",
    "keysource", "enabledoubleencryption", "enablediskencryption",
    "infrastructureencryption", "requireinfrastructureencryption",
    "zoneredundant", "zoneredundancy", "redundancymode", "replication",
    "platformfaultdomaincount",
    "backupretentionintervalinhours", "backupintervalinminutes",
    "backupstorageredundancy",
    "softdeleteretentionindays", "enablesoftdelete", "enablepurgeprotection",
    "ostype", "hypervgeneration", "licensetype",
    "accesstier", "largefilesharesstate", "allowsharedkeyaccess",
    "enablehttpstrafficonly", "supportshttpstrafficonly",
})

# PII / secret key denylist pattern
_KEY_DENY_RE = re.compile(
    r"(secret|password|credential|token|sas|certificate|thumbprint|fingerprint"
    r"|connection|connstr|admin(istrator)?(user|login|name)|private(ip|key|address)"
    r"|publicip|ipaddress|fqdn|hostname|host_name|endpoint|url|uri|email|mail"
    r"|principalid|tenantid|subscriptionid|objectid|clientid|appid"
    r"|customsubdomain|customdomain|key$|^key|accountkey|accesskey"
    r"|primarykey|secondarykey|sharedkey)",
    re.IGNORECASE,
)

# PII / secret value shape patterns
_VALUE_DENY_PATTERNS = [
    re.compile(r"^\d{1,3}(\.\d{1,3}){3}$"),                        # IPv4
    re.compile(r"(?i)^[0-9a-f:]{6,}$"),                              # IPv6-like
    re.compile(r"(?i)^[0-9a-f]{8}-([0-9a-f]{4}-){3}[0-9a-f]{12}$"), # GUID
    re.compile(r"(?i)^https?://"),                                    # URL
    re.compile(r"^[^@]+@[^@]+\.[^@]+$"),                             # email
    re.compile(r"^eyJ[A-Za-z0-9_-]+\."),                             # JWT-like
    re.compile(r"^[A-Za-z0-9+/]{40,}={0,2}$"),                       # base64 blob
]


def _is_pii_key(key: str) -> bool:
    return bool(_KEY_DENY_RE.search(key))


def _is_pii_value(val: str) -> bool:
    return any(p.search(val) for p in _VALUE_DENY_PATTERNS)


def walk_properties(
    node: Any, path: tuple[str, ...] = (), depth: int = 0, max_depth: int = MAX_PROPERTY_DEPTH
):
    """Yield (path_tuple, scalar_value_str) from a nested dict/list, with PII scrubbing."""
    if depth > max_depth:
        return
    if isinstance(node, dict):
        for k, v in node.items():
            k_lower = str(k).lower()
            if _is_pii_key(k_lower):
                continue
            yield from walk_properties(v, path + (k_lower,), depth + 1, max_depth)
    elif isinstance(node, list):
        for item in node:
            yield from walk_properties(item, path, depth + 1, max_depth)
    else:
        val_str = str(node).strip()
        if val_str and not _is_pii_value(val_str):
            yield (path, val_str)


def _insert_aggregation(tree: dict, path: tuple[str, ...], value: str) -> None:
    """Whitelist-gated insertion into nested Counter tree."""
    if not path:
        return
    leaf_name = path[-1]
    if leaf_name not in PROPERTY_LEAF_WHITELIST:
        return
    node = tree
    for segment in path[:-1]:
        node = node.setdefault(segment, {})
    counter = node.setdefault(leaf_name, Counter())
    if isinstance(counter, Counter):
        counter[value] += 1


def _emit_aggregation(tree: dict, total_count: int = 1) -> dict:
    """Convert Counters to {value: fraction} dicts (top-3, fraction of total)."""
    result = {}
    for key, val in tree.items():
        if isinstance(val, Counter):
            top3 = val.most_common(3)
            total = sum(val.values())
            if total > 0:
                result[key] = {v: round(c / total, 3) for v, c in top3}
        elif isinstance(val, dict):
            sub = _emit_aggregation(val, total_count)
            if sub:
                result[key] = sub
    return result


def aggregate_resources(rows: list[dict]) -> dict[str, dict]:
    """Group by type (skipping noise), aggregate properties.
    
    Returns {type: {totalCount, propertyAggregations}}.
    """
    grouped: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        rtype = (row.get("type") or "").lower()
        if not rtype or _is_noise(rtype):
            continue
        grouped[rtype].append(row)

    aggregations: dict[str, dict] = {}
    for rtype, type_rows in sorted(grouped.items()):
        tree: dict = {}
        for row in type_rows:
            # Walk top-level fields: location, kind, sku, identity, properties
            for field in ("location", "kind"):
                val = row.get(field)
                if val and isinstance(val, str):
                    _insert_aggregation(tree, (field,), val.lower())

            sku = row.get("sku")
            if isinstance(sku, dict):
                for p, v in walk_properties(sku, ("sku",)):
                    _insert_aggregation(tree, p, v)

            identity = row.get("identity")
            if isinstance(identity, dict):
                id_type = identity.get("type")
                if id_type and isinstance(id_type, str):
                    _insert_aggregation(tree, ("identity",), id_type)

            props = row.get("properties")
            if isinstance(props, dict):
                for p, v in walk_properties(props, ("properties",)):
                    _insert_aggregation(tree, p, v)

        prop_agg = _emit_aggregation(tree, len(type_rows))
        aggregations[rtype] = {
            "totalCount": len(type_rows),
            "propertyAggregations": prop_agg,
        }

    return aggregations


resource_type_aggregations = aggregate_resources(resources)
print(f"Aggregated {len(resource_type_aggregations)} distinct resource types.")

# Show an example
example_type = next(iter(resource_type_aggregations), None)
if example_type:
    print(f"\nExample — {example_type}:")
    print(json.dumps(resource_type_aggregations[example_type], indent=2))

Aggregated 150 distinct resource types.

Example — microsoft.apimanagement/service:
{
  "totalCount": 16,
  "propertyAggregations": {
    "location": {
      "swedencentral": 0.312,
      "eastus2": 0.188,
      "eastus": 0.125
    },
    "sku": {
      "name": {
        "Premium": 0.625,
        "Developer": 0.188,
        "Consumption": 0.062
      },
      "capacity": {
        "2": 0.5,
        "1": 0.25,
        "3": 0.125
      }
    },
    "properties": {
      "publicnetworkaccess": {
        "Enabled": 1.0
      }
    },
    "identity": {
      "SystemAssigned": 0.75,
      "SystemAssigned, UserAssigned": 0.167,
      "UserAssigned": 0.083
    }
  }
}


## Cell 6: Build FIM Transactions

In [16]:
def build_transactions(rows: list[dict]):
    """Build per-RG transactions of distinct lowercase resource types.

    Each transaction = sorted list of distinct ARM types in one RG.
    Drops auto-created types. Drops length-1 transactions (can't contribute to
    itemsets of length >= 2).

    Returns (transactions, filtered_types, raw_rg_count, single_type_rg_count).
    """
    grouped: dict[tuple[str, str], set[str]] = defaultdict(set)
    filtered: set[str] = set()
    for row in rows:
        sub = row.get("subscriptionId")
        rg = (row.get("resourceGroup") or "").lower()
        rtype = (row.get("type") or "").lower()
        if not sub or not rg or not rtype:
            continue
        if _is_auto_created(rtype):
            filtered.add(rtype)
            continue
        grouped[(sub, rg)].add(rtype)

    raw_rg_count = sum(1 for items in grouped.values() if items)
    transactions = [sorted(items) for items in grouped.values() if len(items) >= 2]
    single_type_rg_count = raw_rg_count - len(transactions)
    return transactions, sorted(filtered), raw_rg_count, single_type_rg_count


transactions, filtered_types, raw_rg_count, single_type_rg_count = build_transactions(resources)

print(f"Raw resource groups with resources: {raw_rg_count}")
print(f"Single-type RGs (excluded):         {single_type_rg_count}")
print(f"Effective transactions (len >= 2):   {len(transactions)}")
print(f"Filtered auto-created types:         {len(filtered_types)}")
if filtered_types:
    print(f"  Examples: {filtered_types[:5]}")

Raw resource groups with resources: 1448
Single-type RGs (excluded):         959
Effective transactions (len >= 2):   489
Filtered auto-created types:         51
  Examples: ['microsoft.alertsmanagement/prometheusrulegroups', 'microsoft.alertsmanagement/smartdetectoralertrules', 'microsoft.automation/automationaccounts/runbooks', 'microsoft.cdn/profiles/afdendpoints', 'microsoft.cloudtest/accounts']


## Cell 7: FP-Growth Mining — Maximal Frequent Itemsets

In [17]:
def _reliability(count: int) -> str:
    """Bucket absolute count into reliability tiers for the LLM."""
    if count >= 30:
        return "high"
    if count >= 10:
        return "medium"
    if count >= 5:
        return "low"
    return "exploratory"


def compute_mfis(itemsets: list[dict]) -> list[dict]:
    """Return Maximal Frequent Itemsets — those with no frequent superset."""
    sets = [(frozenset(it["itemset"]), it) for it in itemsets]
    sets.sort(key=lambda s: (-len(s[0]), -float(s[1].get("support", 0))))
    kept: list[tuple[frozenset, dict]] = []
    for s, entry in sets:
        if any(s < ks for ks, _ in kept):
            continue
        kept.append((s, entry))
    return [entry for _, entry in kept]


# --- Run FP-Growth ---
n = len(transactions)
mfis: list[dict] = []
min_support = 0.0

if n < MIN_TRANSACTIONS:
    print(f"WARNING: Only {n} transactions (need >= {MIN_TRANSACTIONS}). Mining may be unreliable.")

if n > 0:
    min_support = max(MIN_SUPPORT_FLOOR, MIN_ABSOLUTE_COUNT / n)
    print(f"FP-Growth: {n} transactions, min_support={min_support:.4f} "
          f"(absolute threshold={MIN_ABSOLUTE_COUNT})")

    te = TransactionEncoder()
    te_ary = te.fit_transform(transactions)
    df = pd.DataFrame(te_ary, columns=te.columns_)

    freq_df = fpgrowth(df, min_support=min_support, use_colnames=True)
    print(f"  Raw frequent itemsets: {len(freq_df)}")

    # Filter to itemsets with length >= MIN_ITEMSET_LEN
    freq_df = freq_df[freq_df["itemsets"].apply(len) >= MIN_ITEMSET_LEN]
    print(f"  After length filter (>= {MIN_ITEMSET_LEN}): {len(freq_df)}")

    if len(freq_df) > 0:
        all_itemsets = []
        for _, row in freq_df.iterrows():
            sup = float(row["support"].item()) if hasattr(row["support"], "item") else float(row["support"])
            count = int(round(sup * n))
            all_itemsets.append({
                "itemset": sorted(row["itemsets"]),
                "support": round(sup, 4),
                "count": count,
                "reliability": _reliability(count),
            })

        mfis = compute_mfis(all_itemsets)
        if len(mfis) > MAX_ITEMSETS:
            mfis = sorted(mfis, key=lambda x: (-x["count"], -len(x["itemset"])))[:MAX_ITEMSETS]
            print(f"  Capped MFIs to {MAX_ITEMSETS}")

        print(f"  Maximal Frequent Itemsets: {len(mfis)}")
        print(f"\n  Top 5 MFIs:")
        for i, mfi in enumerate(sorted(mfis, key=lambda x: -x["count"])[:5], 1):
            print(f"    {i}. support={mfi['support']:.3f}  count={mfi['count']}  "
                  f"reliability={mfi['reliability']}  {mfi['itemset']}")
    else:
        print("  NOTE: No itemsets of sufficient length found.")
else:
    print("No transactions available for mining.")

FP-Growth: 489 transactions, min_support=0.0200 (absolute threshold=4)
  Raw frequent itemsets: 2617
  After length filter (>= 2): 2572
  Maximal Frequent Itemsets: 76

  Top 5 MFIs:
    1. support=0.049  count=24  reliability=medium  ['microsoft.insights/datacollectionendpoints', 'microsoft.insights/datacollectionrules']
    2. support=0.041  count=20  reliability=medium  ['microsoft.network/networksecuritygroups', 'microsoft.network/privatednszones', 'microsoft.network/privateendpoints', 'microsoft.network/virtualnetworks', 'microsoft.web/certificates', 'microsoft.web/serverfarms', 'microsoft.web/sites']
    3. support=0.029  count=14  reliability=medium  ['microsoft.keyvault/vaults', 'microsoft.managedidentity/userassignedidentities', 'microsoft.network/networksecuritygroups', 'microsoft.network/networksecurityperimeters', 'microsoft.network/publicipaddresses', 'microsoft.network/trafficmanagerprofiles', 'microsoft.network/virtualnetworks']
    4. support=0.027  count=13  reliabilit

## Cell 8: Assemble Data Payload

In [18]:
sub_count = len({r.get("subscriptionId") for r in resources if r.get("subscriptionId")})
rg_count = len({
    (r.get("subscriptionId"), (r.get("resourceGroup") or "").lower())
    for r in resources
    if r.get("subscriptionId") and r.get("resourceGroup")
})

payload = {
    "userQuery": USER_QUERY,
    "resourceContext": {
        "subscriptionCount": sub_count,
        "resourceGroupCount": rg_count,
        "resourceTypes": resource_type_aggregations,
    },
    "tenantPatterns": {
        "transactionUnit": "resourceGroup",
        "algorithm": "fpgrowth",
        "minSupport": round(min_support, 4),
        "reliabilityBuckets": {
            "high": ">=30 RGs",
            "medium": ">=10 RGs",
            "low": ">=5 RGs",
            "exploratory": f">={MIN_ABSOLUTE_COUNT} RGs",
        },
        "maximalFrequentItemsets": mfis,
    },
}

payload_json = json.dumps(payload, ensure_ascii=False)
print(f"Payload size: {len(payload_json):,} chars")
print(f"Subscriptions: {sub_count}")
print(f"Resource groups: {rg_count}")
print(f"Resource types in aggregation: {len(resource_type_aggregations)}")
print(f"MFIs included: {len(mfis)}")

Payload size: 67,957 chars
Subscriptions: 136
Resource groups: 1493
Resource types in aggregation: 150
MFIs included: 76


## Cell 9: Generate Narrative Insights

In [19]:
INSIGHTS_PROMPT = """# Role and Objective
You are an expert Azure Insight Agent. Your mission is to analyze the user's existing infrastructure data and produce insights that inform downstream infrastructure plan generation.

# Input Data Format

You will receive a JSON object with three sections:

## userQuery
The user's infrastructure request. Focus your insights on patterns most relevant to this request, but draw from all tenant-wide data.

## resourceContext
Per-resource-type property aggregations across the tenant.
- Each key is an ARM resource type (e.g. "microsoft.storage/storageaccounts").
- `totalCount`: how many instances of this type exist in the tenant.
- `propertyAggregations`: nested object where each leaf is a dict of `{value: fraction}`.
  - `fraction` is the share of instances that have that value (0.0\u20131.0). The top 3 values are shown; the implied remainder is `1 - sum(fractions)`.
  - Example: `"location": {"eastus": 0.6, "westus2": 0.3}` means 60% of instances are in eastus, 30% in westus2, and 10% elsewhere.

## tenantPatterns
Frequent co-occurrence patterns mined via FP-Growth across all resource groups.
- `maximalFrequentItemsets`: list of Maximal Frequent Itemsets (MFIs).
  - Each MFI is a set of ARM resource types that frequently appear together in the same resource group.
  - `itemset`: the list of ARM types in the pattern.
  - `support`: fraction of resource groups (0.0\u20131.0) that contain ALL types in this itemset.
  - `reliability`: quality tier based on absolute count \u2014 "high" (\u226530 RGs), "medium" (\u226510), "low" (\u22655), "exploratory" (\u22654).
  - An MFI is \"maximal\" because no larger frequent itemset contains it as a subset. Every subset of an MFI is also frequent.
  - Example: `{"itemset": ["microsoft.keyvault/vaults", "microsoft.storage/storageaccounts", "microsoft.network/virtualnetworks"], "support": 0.15, "reliability": "medium"}` means 15% of resource groups contain all three of these types together.

# Process
1. Analyze the property aggregations to identify architectural conventions (SKU choices, security posture, region preferences, redundancy settings).
2. Analyze the MFIs to identify common resource grouping patterns \u2014 which services the tenant typically deploys together.
3. Cross-reference both: e.g., if an MFI shows KeyVault+Storage co-deploy often, and property aggregations show Storage uses LRS 80% of the time, combine into a richer insight.
4. Focus on patterns relevant to the user's query, but include important tenant-wide conventions too.
5. Re-examine your insights for completeness and accuracy.

# Insight Guidelines
When selecting resource properties to base insights on:
- Only consider properties that represent explicit user decisions affecting design.
- Never include properties involving runtime, versions, implementation details, app settings, default values, operational settings, or boilerplate configurations.
- Never include instance-specific properties of a resource.

### Structure of an Insight

Each insight must contain three parts: an observed pattern, the reasoning behind it, and a planning implication.
- The reasoning must be grounded in factual information from the data. Do not make assumptions.
- The planning implication must state concrete actions or decisions for infra planning that align with the user's requirements.
- The reasoning must clearly connect the observed pattern to the planning implication.

### Filtering

Use the following areas as a guide when deciding which resource properties are meaningful:
- Region
- Resource pairing
- Security posture
- Cost
- Naming and tagging conventions
- Azure policies

# Output

Return a JSON object with an \"insights\" key containing an array of insight strings.

```json
{
  "insights": [
    "Insight 1",
    "Insight 2",
    "Insight 3"
  ]
}
```

Each insight must be a single sentence with this structure: \"[observed pattern]: [reasoning] [planning implication]\"."""

print(f"Sending payload ({len(payload_json):,} chars) to Azure OpenAI ...")

response = aoai.chat.completions.create(
    model=AOAI_DEPLOYMENT,
    messages=[
        {"role": "system", "content": INSIGHTS_PROMPT},
        {"role": "user", "content": payload_json},
    ],
    response_format={"type": "json_object"},
    # temperature=0.3,
)

raw_text = response.choices[0].message.content or "{}"
print(f"Response received ({len(raw_text):,} chars).")

# Parse and normalise
parsed = json.loads(raw_text)

insights: list[str] = []
if isinstance(parsed, list):
    insights = [str(x) for x in parsed]
elif isinstance(parsed, dict):
    for key in ("insights", "Insights", "items", "results"):
        if key in parsed and isinstance(parsed[key], list):
            insights = [str(x) for x in parsed[key]]
            break
    if not insights:
        # Try first list-valued key
        for v in parsed.values():
            if isinstance(v, list) and v and isinstance(v[0], str):
                insights = [str(x) for x in v]
                break

print(f"\nExtracted {len(insights)} insights.")

Sending payload (67,957 chars) to Azure OpenAI ...
Response received (3,636 chars).

Extracted 9 insights.


## Cell 10: Save Output

In [20]:
output_path = Path("results/itemset.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(insights, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {len(insights)} insights to {output_path}")
print()
for i, insight in enumerate(insights, 1):
    print(f"{i:2}. {insight}")

Wrote 9 insights to .azure\insights_itemset.json

 1. VNet + NSG + Private Endpoint pattern: MFIs show virtual networks, network security groups, private endpoints and private DNS co-occurring with web apps and certificates (support 0.0409), so plan the finance web app and Azure SQL backend inside the same VNet protected by NSGs and use Private Endpoints with Private DNS to keep all traffic internal.
 2. Key Vault hardening conventions: tenant Key Vaults universally enable purge protection and mostly retain soft-deleted objects for 90 days (enablepurgeprotection True 100%, softdeleteretentionindays 90 = 94.4%), therefore design the finance app to store secrets in Key Vault with purge protection and soft-delete enabled and additionally restrict Key Vault network access (move from broad public access toward firewall/perimeter or VNet rules).
 3. User-assigned managed identities are standard: many resource types use user-assigned identities (storage accounts userAssigned 55%, databases id